In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.cluster import DBSCAN

# List of directories to process
directories = ["Africa", "Asia", "Baikal Lake", "Europe", "India", "Mexico", "North America", "Russia", "South America"]

base_path = "data/Gemstat_Data_Cleaned_Raw"

# Create an empty list to store results from all directories
final_results = []

# Function to assign Data Identifiers based on proximity
def assign_data_identifier(df, eps_km=10):  # Adjust `eps_km` for clustering radius
    if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
        print("Latitude and Longitude columns are required.")
        return df

    # Extract lat/lon
    coords = df[['Latitude', 'Longitude']].dropna().to_numpy()

    if len(coords) == 0:
        df['Data Identifier'] = -1  # Default ID for missing locations
        return df

    # Convert eps from kilometers to radians (Earth radius = 6371 km)
    eps = eps_km / 6371.0

    # Apply DBSCAN with Haversine metric
    dbscan = DBSCAN(eps=eps, min_samples=1, metric='haversine')
    cluster_labels = dbscan.fit_predict(np.radians(coords))

    # Assign Data Identifiers
    df.loc[df[['Latitude', 'Longitude']].notna().all(axis=1), 'Data Identifier'] = cluster_labels

    return df

for dir_name in directories:
    file_path = os.path.join(base_path, dir_name, "cleaned_samples.csv")
    
    # Check if the file exists before reading
    if os.path.exists(file_path):
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Step 1: Count occurrences of each station
        station_counts = df['Station Identifier'].value_counts()
        top_stations = station_counts.nlargest(10).index  # Top 10 stations
        
        # Step 2: Filter the DataFrame to only include top 10 stations
        filtered_df = df[df['Station Identifier'].isin(top_stations)]
        
        # Step 3: Keep only one row per station
        result = filtered_df.groupby('Station Identifier').head(1).copy()
        
        # Step 4: Add additional columns
        result['Total Count'] = result['Station Identifier'].map(station_counts)  # Total occurrences
        result['Continent'] = dir_name  # Continent name

        # Step 5: Assign Data Identifiers based on proximity
        result = assign_data_identifier(result)

        # Append to final results
        final_results.append(result)

# Combine all results into a single DataFrame
final_df = pd.concat(final_results, ignore_index=True)

# Save the final dataset
final_df.to_csv("data/processed_lakes.csv", index=False)
print("Processing complete! Saved as data/processed_lakes.csv")
